# TAWSEEM Dataset Analysis: PROVEDIt DNA Profiling Data

Comprehensive analysis of the PROVEDIt (Process for the Validation of DNA Mixture Interpretation Technology) dataset used in TAWSEEM project for estimating the Number of Contributors (NOC) in DNA mixtures.

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Data Structure Analysis](#data-structure-analysis)  
3. [Sample Distribution](#sample-distribution)
4. [Feature Engineering Analysis](#feature-engineering-analysis)
5. [Quality Assessment](#quality-assessment)
6. [Statistical Summary](#statistical-summary)
7. [Preprocessing Pipeline](#preprocessing-pipeline)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Dataset Overview

The PROVEDIt dataset contains DNA electropherogram data from controlled DNA mixtures with **known contributors (1-5 persons)**. The data is organized by:

- **Number of Contributors**: 1-Person, 2-Person, 3-Person, 4-Person, 5-Person
- **STR Kits**: IDPlus, PP16HS, F6C, GF (different forensic DNA typing kits)
- **Time Intervals**: 5sec, 10sec, 15sec, 20sec, 25sec (injection time variations)
- **Sample Types**: Various ratios and degradation conditions

This is a **supervised classification problem** where we predict the Number of Contributors (NOC) from STR electropherogram features.

In [ ]:
# Define dataset paths
base_path = Path("PROVEDIt_1-5-Person CSVs Filtered")
processed_path = Path("data/processed")

# Get all dataset subdirectories
if base_path.exists():
    dataset_dirs = [d for d in base_path.iterdir() if d.is_dir()]
    print(f"Found {len(dataset_dirs)} dataset subdirectories:")
    for i, dir_name in enumerate(dataset_dirs, 1):
        print(f"  {i}. {dir_name.name}")
else:
    print(f"Dataset directory not found: {base_path}")

# Check processed data
if processed_path.exists():
    processed_files = list(processed_path.glob('*.csv'))
    print(f"\nProcessed files ({len(processed_files)}):")
    for file in processed_files:
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"  - {file.name:<25} ({size_mb:.1f} MB)")
else:
    print(f"\nProcessed data directory not found: {processed_path}")

## 2. Data Structure Analysis

### Raw Data Format

Each CSV file contains STR electropherogram data with the following structure:
- **Sample File**: Unique identifier for each DNA profile
- **Marker**: STR loci (e.g., D8S1179, D21S11, TH01, etc.)
- **Dye**: Fluorescent dye channel (B=Blue, G=Green, Y=Yellow, R=Red)
- **Allele 1-100**: Peak information (Allele, Size in bp, Height in RFU)
- **OL**: Off-ladder alleles (outside standard allelic ladder)

In [ ]:
# Load a sample raw data file to examine structure
sample_file = "PROVEDIt_1-5-Person CSVs Filtered/PROVEDIt_1-5-Person CSVs Filtered_3130_IDPlus28cycles/1-Person/10 sec/RD14-0003_IP_10sec_GM_F_1P.csv"

if os.path.exists(sample_file):
    # Read first few rows to understand structure
    sample_df = pd.read_csv(sample_file, nrows=10)
    
    print("Raw Data Structure:")
    print(f"File: {sample_file}")
    print(f"Shape: {sample_df.shape}")
    print(f"\nColumns ({len(sample_df.columns)}): ")
    
    # Group columns by type
    basic_cols = ['Sample File', 'Marker', 'Dye']
    allele_cols = [col for col in sample_df.columns if col.startswith('Allele')]
    size_cols = [col for col in sample_df.columns if col.startswith('Size')]
    height_cols = [col for col in sample_df.columns if col.startswith('Height')]
    
    print(f"  Basic info: {basic_cols}")
    print(f"  Allele columns: {len(allele_cols)} (Allele 1 to Allele {len(allele_cols)})")
    print(f"  Size columns: {len(size_cols)} (Size 1 to Size {len(size_cols)})")
    print(f"  Height columns: {len(height_cols)} (Height 1 to Height {len(height_cols)})")
    
    print(f"\nFirst 5 rows:")
    display(sample_df[['Sample File', 'Marker', 'Dye', 'Allele 1', 'Size 1', 'Height 1', 'Allele 2', 'Size 2', 'Height 2']].head())
else:
    print(f"Sample file not found: {sample_file}")

### Processed Data Format

After preprocessing, the data is transformed into profile-level features:
- **Allele/Size/Height features**: Up to 9 peaks per locus
- **OL indicators**: Binary flags for Off-Ladder alleles  
- **Missing indicators**: Binary flags for missing data
- **Metadata**: Dye, Marker, profile_loci information
- **Target**: NOC (Number of Contributors) - 1 to 5

In [ ]:
# Load processed data for analysis
processed_files = {
    'single': 'data/processed/single_processed.csv',
    'four_union': 'data/processed/four_union_processed.csv'
}

dfs = {}
for name, file_path in processed_files.items():
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        dfs[name] = df
        print(f"\n{name.upper()} Dataset:")
        print(f"  Shape: {df.shape}")
        print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
        
        # Check NOC distribution
        noc_dist = df['NOC'].value_counts().sort_index()
        print(f"  NOC distribution: {dict(noc_dist)}")
    else:
        print(f"File not found: {file_path}")

# Detailed column analysis for the largest dataset
if dfs:
    main_df = list(dfs.values())[0]
    
    print(f"\nColumn Categories:")
    
    # Group columns by type
    allele_cols = [col for col in main_df.columns if col.startswith('Allele')]
    size_cols = [col for col in main_df.columns if col.startswith('Size')]
    height_cols = [col for col in main_df.columns if col.startswith('Height')]
    ol_cols = [col for col in main_df.columns if col.startswith('OL_ind')]
    missing_cols = [col for col in main_df.columns if col.startswith('Missing')]
    meta_cols = ['Dye', 'Marker', 'profile_loci', 'NOC']
    
    print(f"  Allele features: {len(allele_cols)} columns")
    print(f"  Size features: {len(size_cols)} columns")
    print(f"  Height features: {len(height_cols)} columns")
    print(f"  Off-Ladder indicators: {len(ol_cols)} columns")
    print(f"  Missing indicators: {len(missing_cols)} columns")
    print(f"  Metadata: {len(meta_cols)} columns")
    print(f"  Total: {len(main_df.columns)} columns")
    
    # Feature matrix dimensions
    feature_cols = allele_cols + size_cols + height_cols + ol_cols + missing_cols
    print(f"\nFeature Matrix: {len(main_df)} samples × {len(feature_cols)} features")

## 3. Sample Distribution Analysis

Understanding the distribution of samples across different categories is crucial for model performance and bias assessment.

In [ ]:
# Analyze NOC distribution across datasets
fig, axes = plt.subplots(1, len(dfs), figsize=(5 * len(dfs), 6))
if len(dfs) == 1:
    axes = [axes]

for i, (name, df) in enumerate(dfs.items()):
    noc_counts = df['NOC'].value_counts().sort_index()
    
    ax = axes[i] if len(dfs) > 1 else axes[0]
    bars = ax.bar(noc_counts.index, noc_counts.values, alpha=0.8, color='skyblue', edgecolor='navy')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'NOC Distribution - {name.title()}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Contributors (NOC)', fontsize=12)
    ax.set_ylabel('Number of Profiles', fontsize=12)
    ax.set_xticks(range(1, 6))
    ax.grid(axis='y', alpha=0.3)
    
    # Calculate percentages
    total = noc_counts.sum()
    percentages = noc_counts / total * 100
    
    print(f"\n{name.upper()} NOC Distribution:")
    for noc, count in noc_counts.items():
        print(f"  {noc}-Person: {count:,} profiles ({percentages[noc]:.1f}%)")
    print(f"  Total: {total:,} profiles")

plt.tight_layout()
plt.show()

# Class balance analysis
print("\nClass Balance Assessment:")
for name, df in dfs.items():
    noc_counts = df['NOC'].value_counts().sort_index()
    min_class = noc_counts.min()
    max_class = noc_counts.max()
    balance_ratio = max_class / min_class
    
    print(f"\n{name.title()}:")
    print(f"  Min class size: {min_class:,}")
    print(f"  Max class size: {max_class:,}")
    print(f"  Imbalance ratio: {balance_ratio:.2f}:1")
    
    if balance_ratio <= 2:
        print(f"  Status: Well balanced")
    elif balance_ratio <= 5:
        print(f"  Status: Moderately imbalanced")
    else:
        print(f"  Status: Highly imbalanced - may need balancing")

In [ ]:
# Analyze STR markers and dye channels
if dfs:
    main_df = list(dfs.values())[0]
    
    # Marker distribution
    marker_counts = main_df['Marker'].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # STR Markers
    marker_counts.plot(kind='bar', ax=ax1, color='lightcoral', alpha=0.8)
    ax1.set_title('STR Marker Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('STR Markers', fontsize=12)
    ax1.set_ylabel('Number of Loci', fontsize=12)
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    # Dye channels
    dye_counts = main_df['Dye'].value_counts()
    colors = {'B': 'blue', 'G': 'green', 'Y': 'gold', 'R': 'red'}
    dye_colors = [colors.get(dye, 'gray') for dye in dye_counts.index]
    
    bars = ax2.bar(dye_counts.index, dye_counts.values, color=dye_colors, alpha=0.7)
    ax2.set_title('Dye Channel Distribution', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Dye Channels', fontsize=12)
    ax2.set_ylabel('Number of Loci', fontsize=12)
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSTR Kit Analysis:")
    print(f"  Total unique markers: {len(marker_counts)}")
    print(f"  Markers: {list(marker_counts.index)}")
    
    print(f"\nDye Channel Analysis:")
    dye_names = {'B': 'Blue', 'G': 'Green', 'Y': 'Yellow', 'R': 'Red'}
    for dye, count in dye_counts.items():
        full_name = dye_names.get(dye, f'Unknown ({dye})')
        print(f"  {dye} ({full_name}): {count} markers")

## 4. Feature Engineering Analysis

Understanding the engineered features and their characteristics is essential for model interpretation and performance optimization.

In [ ]:
if dfs:
    main_df = list(dfs.values())[0]
    
    # Separate numeric features
    numeric_cols = [col for col in main_df.columns if col not in ['Dye', 'Marker', 'NOC']]
    
    # Feature type analysis
    feature_types = {
        'Allele': [col for col in numeric_cols if col.startswith('Allele')],
        'Size': [col for col in numeric_cols if col.startswith('Size')],
        'Height': [col for col in numeric_cols if col.startswith('Height')],
        'OL_indicators': [col for col in numeric_cols if col.startswith('OL_ind')],
        'Missing_indicators': [col for col in numeric_cols if col.startswith('Missing')],
        'Other': [col for col in numeric_cols if not any(col.startswith(prefix) for prefix in ['Allele', 'Size', 'Height', 'OL_ind', 'Missing'])]
    }
    
    print("Feature Categories Summary:")
    total_features = 0
    for category, features in feature_types.items():
        if features:
            print(f"  {category}: {len(features)} features")
            total_features += len(features)
    
    print(f"\nTotal numeric features: {total_features}")
    
    # Statistical summary for each feature type
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (category, features) in enumerate(feature_types.items()):
        if features and i < len(axes):
            # Calculate basic statistics
            feature_data = main_df[features]
            
            if category in ['OL_indicators', 'Missing_indicators']:
                # Binary features - show proportion of 1s
                proportions = feature_data.mean()
                axes[i].hist(proportions, bins=20, alpha=0.7, color='lightgreen', edgecolor='darkgreen')
                axes[i].set_title(f'{category}\nProportion of Positive Cases', fontweight='bold')
                axes[i].set_xlabel('Proportion')
            else:
                # Continuous features - show distribution of means
                means = feature_data.mean()
                axes[i].hist(means, bins=20, alpha=0.7, color='lightblue', edgecolor='navy')
                axes[i].set_title(f'{category}\nFeature Mean Distribution', fontweight='bold')
                axes[i].set_xlabel('Mean Value')
            
            axes[i].set_ylabel('Frequency')
            axes[i].grid(axis='y', alpha=0.3)
    
    # Remove empty subplots
    for j in range(len(feature_types), len(axes)):
        axes[j].remove()
    
    plt.tight_layout()
    plt.show()
    
    # Missing data analysis
    print(f"\nMissing Data Analysis:")
    missing_stats = main_df[numeric_cols].isnull().sum()
    total_missing = missing_stats.sum()
    
    if total_missing > 0:
        print(f"  Total missing values: {total_missing:,}")
        print(f"  Features with missing data: {(missing_stats > 0).sum()}")
        
        if (missing_stats > 0).any():
            print(f"  Top features with missing data:")
            top_missing = missing_stats[missing_stats > 0].sort_values(ascending=False).head(10)
            for feature, count in top_missing.items():
                pct = count / len(main_df) * 100
                print(f"    {feature}: {count:,} ({pct:.1f}%)")
    else:
        print(f"  No missing values detected in processed data")

In [ ]:
# Peak height and size analysis
if dfs:
    main_df = list(dfs.values())[0]
    
    # Analyze height distributions by NOC
    height_cols = [col for col in main_df.columns if col.startswith('Height')]
    
    if height_cols:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Height distribution by NOC
        for noc in sorted(main_df['NOC'].unique()):
            noc_data = main_df[main_df['NOC'] == noc]
            heights = noc_data[height_cols].values.flatten()
            heights = heights[heights > 0]  # Remove zeros/missing
            
            if len(heights) > 0:
                ax1.hist(np.log10(heights + 1), alpha=0.6, bins=50, 
                        label=f'{noc}-Person', density=True)
        
        ax1.set_title('Peak Height Distribution by NOC\n(Log10 scale)', fontweight='bold')
        ax1.set_xlabel('Log10(Height + 1)')
        ax1.set_ylabel('Density')
        ax1.legend()
        ax1.grid(alpha=0.3)
        
        # Average number of peaks per profile by NOC
        noc_peak_counts = []
        noc_values = []
        
        for noc in sorted(main_df['NOC'].unique()):
            noc_data = main_df[main_df['NOC'] == noc]
            # Count non-zero heights per profile
            peak_counts = (noc_data[height_cols] > 0).sum(axis=1)
            noc_peak_counts.extend(peak_counts.tolist())
            noc_values.extend([noc] * len(peak_counts))
        
        peak_df = pd.DataFrame({'NOC': noc_values, 'Peak_Count': noc_peak_counts})
        
        # Box plot of peak counts
        sns.boxplot(data=peak_df, x='NOC', y='Peak_Count', ax=ax2)
        ax2.set_title('Number of Peaks per Profile by NOC', fontweight='bold')
        ax2.set_xlabel('Number of Contributors (NOC)')
        ax2.set_ylabel('Number of Peaks per Profile')
        ax2.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Statistical summary
        print("Peak Analysis Summary:")
        peak_stats = peak_df.groupby('NOC')['Peak_Count'].agg(['mean', 'std', 'min', 'max'])
        print(peak_stats.round(2))
        
        # Height statistics by NOC
        print(f"\nHeight Statistics by NOC (RFU units):")
        for noc in sorted(main_df['NOC'].unique()):
            noc_data = main_df[main_df['NOC'] == noc]
            heights = noc_data[height_cols].values.flatten()
            heights = heights[heights > 0]
            
            if len(heights) > 0:
                print(f"  {noc}-Person: Mean={heights.mean():.0f}, Median={np.median(heights):.0f}, "
                      f"Max={heights.max():.0f}, Count={len(heights):,}")

## 5. Quality Assessment

Evaluating data quality indicators including Off-Ladder alleles, missing data patterns, and potential artifacts.

In [ ]:
if dfs:
    main_df = list(dfs.values())[0]
    
    # Off-Ladder analysis
    ol_cols = [col for col in main_df.columns if col.startswith('OL_ind')]
    missing_cols = [col for col in main_df.columns if col.startswith('Missing')]
    
    if ol_cols and missing_cols:
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Off-Ladder frequency by NOC
        ol_by_noc = []
        for noc in sorted(main_df['NOC'].unique()):
            noc_data = main_df[main_df['NOC'] == noc]
            ol_rate = noc_data[ol_cols].mean().mean()  # Average across all OL indicators
            ol_by_noc.append(ol_rate)
        
        ax1.bar(sorted(main_df['NOC'].unique()), ol_by_noc, alpha=0.7, color='orange')
        ax1.set_title('Off-Ladder Allele Rate by NOC', fontweight='bold')
        ax1.set_xlabel('Number of Contributors')
        ax1.set_ylabel('OL Allele Rate')
        ax1.grid(axis='y', alpha=0.3)
        
        # 2. Missing data frequency by NOC
        missing_by_noc = []
        for noc in sorted(main_df['NOC'].unique()):
            noc_data = main_df[main_df['NOC'] == noc]
            missing_rate = noc_data[missing_cols].mean().mean()
            missing_by_noc.append(missing_rate)
        
        ax2.bar(sorted(main_df['NOC'].unique()), missing_by_noc, alpha=0.7, color='red')
        ax2.set_title('Missing Data Rate by NOC', fontweight='bold')
        ax2.set_xlabel('Number of Contributors')
        ax2.set_ylabel('Missing Data Rate')
        ax2.grid(axis='y', alpha=0.3)
        
        # 3. OL distribution across peak positions
        ol_means = main_df[ol_cols].mean()
        ax3.bar(range(len(ol_means)), ol_means.values, alpha=0.7, color='gold')
        ax3.set_title('OL Frequency by Peak Position', fontweight='bold')
        ax3.set_xlabel('Peak Position')
        ax3.set_ylabel('OL Frequency')
        ax3.set_xticks(range(len(ol_means)))
        ax3.set_xticklabels([f'P{i+1}' for i in range(len(ol_means))])
        ax3.grid(axis='y', alpha=0.3)
        
        # 4. Missing data distribution across peak positions
        missing_means = main_df[missing_cols].mean()
        ax4.bar(range(len(missing_means)), missing_means.values, alpha=0.7, color='crimson')
        ax4.set_title('Missing Data by Peak Position', fontweight='bold')
        ax4.set_xlabel('Peak Position')
        ax4.set_ylabel('Missing Data Frequency')
        ax4.set_xticks(range(len(missing_means)))
        ax4.set_xticklabels([f'P{i+1}' for i in range(len(missing_means))])
        ax4.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Quality metrics summary
        print("Data Quality Summary:")
        
        total_ol_rate = main_df[ol_cols].mean().mean()
        total_missing_rate = main_df[missing_cols].mean().mean()
        
        print(f"\nOverall Quality Metrics:")
        print(f"  Off-Ladder rate: {total_ol_rate:.3f} ({total_ol_rate*100:.1f}%)")
        print(f"  Missing data rate: {total_missing_rate:.3f} ({total_missing_rate*100:.1f}%)")
        
        print(f"\nQuality by NOC:")
        for i, noc in enumerate(sorted(main_df['NOC'].unique())):
            print(f"  {noc}-Person: OL={ol_by_noc[i]:.3f}, Missing={missing_by_noc[i]:.3f}")
        
        # Peak position analysis
        print(f"\nPeak Position Analysis:")
        print(f"  Positions with highest OL rate: {ol_means.nlargest(3).index.tolist()}")
        print(f"  Positions with highest missing rate: {missing_means.nlargest(3).index.tolist()}")

## 6. Statistical Summary

Comprehensive statistical overview of the dataset features and their relationships.

In [ ]:
if dfs:
    main_df = list(dfs.values())[0]
    
    # Select representative features for correlation analysis
    height_cols = [col for col in main_df.columns if col.startswith('Height')][:9]  # First 9 heights
    ol_cols = [col for col in main_df.columns if col.startswith('OL_ind')][:5]     # First 5 OL indicators
    missing_cols = [col for col in main_df.columns if col.startswith('Missing')][:5] # First 5 missing indicators
    
    analysis_cols = height_cols + ol_cols + missing_cols + ['NOC']
    
    if len(analysis_cols) > 1:
        # Correlation matrix
        corr_matrix = main_df[analysis_cols].corr()
        
        # Plot correlation heatmap
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
        
        # Full correlation matrix
        sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
                    square=True, ax=ax1, cbar_kws={'shrink': 0.8})
        ax1.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=14)
        
        # NOC correlations specifically
        noc_corrs = corr_matrix['NOC'].drop('NOC').abs().sort_values(ascending=True)
        
        noc_corrs.plot(kind='barh', ax=ax2, color='lightcoral', alpha=0.7)
        ax2.set_title('Features Most Correlated with NOC\n(Absolute Correlation)', fontweight='bold')
        ax2.set_xlabel('Absolute Correlation with NOC')
        ax2.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"Feature-NOC Correlation Analysis:")
        print(f"\nTop 10 features most correlated with NOC:")
        
        top_corrs = corr_matrix['NOC'].drop('NOC').abs().nlargest(10)
        for feature, corr in top_corrs.items():
            original_corr = corr_matrix.loc[feature, 'NOC']
            direction = "positive" if original_corr > 0 else "negative"
            print(f"  {feature}: {abs(original_corr):.3f} ({direction})")
    
    # Feature variability analysis
    numeric_features = main_df.select_dtypes(include=[np.number]).columns.tolist()
    if 'NOC' in numeric_features:
        numeric_features.remove('NOC')
    
    if numeric_features:
        feature_stats = main_df[numeric_features].agg(['mean', 'std', 'min', 'max', 'skew'])
        
        print(f"\nFeature Variability Summary:")
        print(f"  Total numeric features: {len(numeric_features)}")
        
        # Features with high variability (CV > 1.0)
        cv = feature_stats.loc['std'] / (feature_stats.loc['mean'] + 1e-10)  # Coefficient of variation
        high_var_features = cv[cv > 1.0]
        
        print(f"  High variability features (CV > 1.0): {len(high_var_features)}")
        print(f"  Low variability features (CV ≤ 0.1): {len(cv[cv <= 0.1])}")
        
        # Skewness analysis
        skew_stats = feature_stats.loc['skew']
        highly_skewed = skew_stats[abs(skew_stats) > 2]
        
        print(f"  Highly skewed features (|skew| > 2): {len(highly_skewed)}")
        
        if len(highly_skewed) > 0:
            print(f"  Examples of highly skewed features:")
            for feature, skew_val in highly_skewed.head().items():
                direction = "right" if skew_val > 0 else "left"
                print(f"    {feature}: {skew_val:.2f} ({direction}-skewed)")

## 7. Preprocessing Pipeline Summary

Understanding the data transformation pipeline from raw electropherogram data to machine learning features.

In [ ]:
# Pipeline summary and recommendations
print("TAWSEEM Data Preprocessing Pipeline Summary")
print("="*60)

print("\n1. RAW DATA INGESTION:")
print("   - Input: STR electropherogram CSV files")
print("   - Format: Peak data (Allele, Size, Height) up to 100 peaks per locus")
print("   - Organization: By NOC (1-5), STR kit, and injection time")

print("\n2. FEATURE EXTRACTION:")
print("   - Peak selection: Top 9 peaks per locus (by height)")
print("   - Features per peak: Allele value, Size (bp), Height (RFU)")
print("   - Quality indicators: Off-Ladder and Missing data flags")
print("   - Metadata: Dye channel, STR marker, locus information")

print("\n3. DATA TRANSFORMATION:")
print("   - Vectorized padding: Standardized 9-peak representation")
print("   - Missing value handling: Explicit missing indicators")
print("   - Profile-level aggregation: Locus-based feature vectors")
print("   - Target encoding: NOC as 0-4 (for 1-5 contributors)")

print("\n4. QUALITY CONTROL:")
if dfs:
    main_df = list(dfs.values())[0]
    
    # Calculate quality metrics
    ol_cols = [col for col in main_df.columns if col.startswith('OL_ind')]
    missing_cols = [col for col in main_df.columns if col.startswith('Missing')]
    
    if ol_cols and missing_cols:
        ol_rate = main_df[ol_cols].mean().mean()
        missing_rate = main_df[missing_cols].mean().mean()
        
        print(f"   - Off-Ladder allele rate: {ol_rate:.3f} ({ol_rate*100:.1f}%)")
        print(f"   - Missing data rate: {missing_rate:.3f} ({missing_rate*100:.1f}%)")
    
    # Feature dimensions
    numeric_features = main_df.select_dtypes(include=[np.number]).columns.tolist()
    if 'NOC' in numeric_features:
        numeric_features.remove('NOC')
    
    print(f"   - Final feature dimensions: {len(main_df)} profiles × {len(numeric_features)} features")
    
    # Class distribution
    noc_dist = main_df['NOC'].value_counts().sort_index()
    balance_ratio = noc_dist.max() / noc_dist.min()
    print(f"   - Class balance ratio: {balance_ratio:.1f}:1")

print("\n5. MACHINE LEARNING PIPELINE:")
print("   - Train/Test split: 90/10 stratified by NOC")
print("   - Feature scaling: MinMax normalization (0-1 range)")
print("   - Class balancing: Downsampling for balanced training")
print("   - Cross-validation: 5-fold stratified CV for model selection")

print("\n6. MODEL ARCHITECTURES:")
print("   - MLP (PyTorch): Multi-layer perceptron with dropout")
print("   - XGBoost: Gradient boosting with profile-level features")
print("   - Random Forest: Ensemble method with feature importance")

print("\n7. EVALUATION METRICS:")
print("   - Primary: Multi-class accuracy (1-5 contributors)")
print("   - Secondary: Per-class precision, recall, F1-score")
print("   - ROC Analysis: AUC for each class (One-vs-Rest)")
print("   - Confusion matrices: Detailed misclassification patterns")

print(f"\n{'='*60}")
print("Dataset Analysis Complete!")
print(f"{'='*60}")

# Final recommendations
if dfs:
    print("\nKEY FINDINGS & RECOMMENDATIONS:")
    
    for name, df in dfs.items():
        noc_dist = df['NOC'].value_counts().sort_index()
        balance_ratio = noc_dist.max() / noc_dist.min()
        
        print(f"\n{name.upper()} Dataset:")
        print(f"  • Total profiles: {len(df):,}")
        print(f"  • Feature dimensions: {len([c for c in df.columns if c != 'NOC'])}-D")
        
        if balance_ratio > 3:
            print(f"  • RECOMMENDATION: Apply class balancing (ratio {balance_ratio:.1f}:1)")
        else:
            print(f"  • Class distribution: Well balanced ({balance_ratio:.1f}:1)")
        
        # Data quality assessment
        ol_cols = [col for col in df.columns if col.startswith('OL_ind')]
        if ol_cols:
            ol_rate = df[ol_cols].mean().mean()
            if ol_rate > 0.1:
                print(f"  • ATTENTION: High OL rate ({ol_rate:.2f}) - consider OL handling")
            else:
                print(f"  • Data quality: Good (OL rate: {ol_rate:.3f})")
        
        print(f"  • Suitable for: All ML algorithms (MLP, XGBoost, Random Forest)")